# German Credit 데이터셋 탐색적 데이터 분석 (EDA) — 통합본

---

**데이터셋**: German Credit Dataset (Statlog)  
**출처**: UCI Machine Learning Repository / OpenML  
**날짜**: 2026-03-25  
**과목**: 조상구 교수님 — 3주차 과제

> 해당 노트북은 6개 섹션을 모두 포함한 **통합본**입니다.  
> 섹션별 독립 노트북은 `01_introduction.ipynb` ~ `06_conclusion.ipynb`를 참조하세요.

---

## 목차

1. [서론 (Introduction)](#1.-서론-(Introduction))
2. [데이터 프로파일링 및 기초 탐색 (Data Profiling)](#2.-데이터-프로파일링-및-기초-탐색-(Data-Profiling))
3. [변수별 개별 특성 분석 (Univariate Analysis)](#3.-변수별-개별-특성-분석-(Univariate-Analysis))
4. [상관관계 및 관계 분석 (Multivariate Analysis)](#4.-상관관계-및-관계-분석-(Multivariate-Analysis))
5. [핵심 인사이트 및 가설 검정 (Key Insights)](#5.-핵심-인사이트-및-가설-검정-(Key-Insights))
6. [결론 및 향후 방향 (Conclusion & Recommendation)](#6.-결론-및-향후-방향-(Conclusion-&-Recommendation))

## 라이브러리 및 환경 설정

In [ ]:
# === 라이브러리 임포트 ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.io import arff
from scipy import stats
from scipy.stats import chi2_contingency, mannwhitneyu, shapiro, kruskal
import warnings
warnings.filterwarnings('ignore')

# === 시각화 설정 ===
plt.rcParams['font.family'] = 'Malgun Gothic'   # Windows 한글 폰트
plt.rcParams['axes.unicode_minus'] = False        # 마이너스 부호 깨짐 방지
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')
sns.set_palette('Set2')

# 컬러 팔레트 정의
COLORS = {'good': '#2ecc71', 'bad': '#e74c3c'}
CLASS_PALETTE = [COLORS['good'], COLORS['bad']]

print('환경 설정 완료')

## 데이터 로딩

In [ ]:
# === .arff 파일 로딩 ===
data_raw, meta = arff.loadarff('dataset_31_credit-g.arff')

# numpy structured array → pandas DataFrame 변환
df = pd.DataFrame(data_raw)

# bytes → str 변환 (scipy.io.arff는 범주형을 bytes로 읽음)
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].apply(lambda x: x.decode('utf-8') if isinstance(x, bytes) else x)

print(f'데이터 로딩 완료: {df.shape[0]}행 × {df.shape[1]}열')
df.head(3)

In [ ]:
# === 변수 분류 ===
# 수치형 변수
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
# 범주형 변수 (타겟 제외)
categorical_cols = [c for c in df.select_dtypes(include=['object']).columns if c != 'class']
# 타겟 변수
target = 'class'

print(f'수치형 변수 ({len(numeric_cols)}개): {numeric_cols}')
print(f'범주형 변수 ({len(categorical_cols)}개): {categorical_cols}')
print(f'타겟 변수: {target}')

---

# 1. 서론 (Introduction)

## 1.1 분석 배경 및 목적

### 비즈니스 맥락

금융 기관의 **신용 리스크 평가(Credit Risk Assessment)**는 대출 심사의 핵심 프로세스입니다.  
대출 신청자의 신용도를 정확히 평가하지 못하면 두 가지 유형의 비용이 발생합니다:

| 오류 유형 | 설명 | 비용 |
|-----------|------|------|
| **Type I (FP)** | 신용이 좋은 고객을 나쁘다고 판단 → 대출 거절 | 기회비용 (이자 수익 손실) |
| **Type II (FN)** | 신용이 나쁜 고객을 좋다고 판단 → 대출 승인 | 직접 손실 (원금 미회수) |

### 비대칭 비용 행렬 (Asymmetric Cost Matrix)

이 데이터셋은 **비대칭 비용 구조**를 명시합니다:

```
              예측: Good    예측: Bad
실제: Good      0            1
실제: Bad       5            0
```

- **FN 비용(5)이 FP 비용(1)의 5배**: 부실 대출 승인이 우량 고객 거절보다 5배 더 큰 손실  
- 이는 실제 금융 도메인의 현실을 반영합니다

### 분석 목적

본 분석의 목적은 German Credit 데이터셋에 대한 **체계적 탐색적 데이터 분석(EDA)**을 수행하여:

1. 데이터의 구조와 특성을 깊이 이해하고
2. 신용 등급(good/bad)에 영향을 미치는 핵심 변수를 식별하며
3. 향후 머신러닝 모델 개발을 위한 데이터 기반 인사이트를 도출하는 것입니다.

## 1.2 데이터 셋 설명

| 항목 | 내용 |
|------|------|
| **데이터셋 명** | German Credit Data (Statlog) |
| **출처** | UCI Machine Learning Repository / OpenML (dataset ID: 31) |
| **제공자** | Professor Dr. Hans Hofmann, Universität Hamburg |
| **수집 시기** | 1994년 |
| **레코드 수** | 1,000건 |
| **변수 수** | 21개 (20개 입력 + 1개 타겟) |
| **타겟 변수** | class (good / bad) |
| **도메인** | 금융 — 개인 신용 대출 심사 |
| **데이터 형식** | ARFF (Attribute-Relation File Format) |

### 변수 구성

| 구분 | 변수 수 | 변수명 |
|------|---------|--------|
| 수치형 | 7개 | duration, credit_amount, installment_commitment, residence_since, age, existing_credits, num_dependents |
| 범주형 | 13개 | checking_status, credit_history, purpose, savings_status, employment, personal_status, other_parties, property_magnitude, other_payment_plans, housing, job, own_telephone, foreign_worker |
| 타겟 | 1개 | class (good / bad) |

## 1.3 주요 분석 질문 (Key Questions) 및 가설 설정

본 분석에서 검증하고자 하는 핵심 가설은 다음과 같습니다:

### 가설 (Hypotheses)

| 번호 | 가설 | 유형 | 관련 변수 |
|------|------|------|----------|
| **H1** | 당좌예금 상태(checking_status)가 신용등급에 유의미한 영향을 미친다 | 범주형 × 타겟 | checking_status → class |
| **H2** | 대출 금액(credit_amount)이 클수록 부실(bad) 비율이 증가한다 | 수치형 × 타겟 | credit_amount → class |
| **H3** | 고용 기간(employment)이 길수록 우량(good) 비율이 높다 | 범주형 × 타겟 | employment → class |
| **H4** | 대출 기간(duration)과 대출 금액(credit_amount)은 양의 상관관계를 보인다 | 수치형 × 수치형 | duration ↔ credit_amount |

### 탐색 질문

- 어떤 변수 조합이 신용 등급을 가장 잘 구별하는가?
- 데이터에 숨겨진 세그먼트(예: VIP 고객 vs 일반 고객)가 존재하는가?
- 비대칭 비용 구조를 고려할 때, 어떤 특성이 부실 대출(bad)의 조기 탐지에 유용한가?

---

# 2. 데이터 프로파일링 및 기초 탐색 (Data Profiling)

## 2.1 데이터 명세 확인

전체 레코드 수, 컬럼 수, 변수 타입(Dtype)을 확인하여 데이터의 전체적인 구조를 파악합니다.

In [ ]:
# === 데이터 기본 정보 ===
print('=' * 60)
print('데이터 기본 정보')
print('=' * 60)
print(f'행 수 (레코드): {df.shape[0]:,}')
print(f'열 수 (변수):   {df.shape[1]}')
print(f'메모리 사용량:  {df.memory_usage(deep=True).sum() / 1024:.1f} KB')
print()
df.info()

In [ ]:
# === 컬럼별 상세 명세 표 ===
spec = pd.DataFrame({
    '변수명': df.columns,
    'Dtype': df.dtypes.values,
    '고유값 수': df.nunique().values,
    '결측 수': df.isnull().sum().values,
    '결측률(%)': (df.isnull().sum() / len(df) * 100).round(2).values,
    '예시값': [df[c].iloc[0] for c in df.columns]
})
spec.index = range(1, len(spec) + 1)
spec.index.name = 'No.'
spec

### 해석

- 전체 **1,000건 × 21개 변수**로 구성된 중소규모 데이터셋
- `float64` 타입 7개(수치형), `object` 타입 14개(범주형 + 타겟)
- 각 변수의 고유값 수를 통해 범주형 변수의 카디널리티(cardinality)를 파악할 수 있음

## 2.2 결측치(Missing Value) 분석

결측치의 존재 여부와 패턴을 확인하고, 필요시 처리 전략을 수립합니다.

In [ ]:
# === 결측치 분석 ===
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    '결측 수': missing,
    '결측률(%)': missing_pct
}).sort_values('결측 수', ascending=False)

print('결측치 요약:')
print(f'  - 결측치가 있는 변수: {(missing > 0).sum()}개')
print(f'  - 전체 결측 셀 수: {missing.sum():,}')
print(f'  - 전체 결측률: {missing.sum() / df.size * 100:.2f}%')

if missing.sum() == 0:
    print('\n✅ 결측치가 전혀 없는 완전한 데이터셋입니다!')
else:
    display(missing_df[missing_df['결측 수'] > 0])

In [ ]:
# === 결측치 히트맵 시각화 ===
fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(df.isnull().T, cbar=True, cmap='YlOrRd', yticklabels=True, ax=ax)
ax.set_title('결측치 히트맵 (Missing Value Heatmap)', fontsize=14, fontweight='bold')
ax.set_xlabel('레코드 인덱스')
ax.set_ylabel('변수')
plt.tight_layout()
plt.show()

### 해석

- German Credit 데이터셋은 **결측치가 없는 완전한 데이터셋**입니다
- 이는 UCI Repository에서 이미 전처리된 상태로 제공되었기 때문
- 결측치 처리 전략(제거, 대체, 유지)은 본 데이터셋에서는 불필요하지만,  
  실무에서는 결측치 패턴이 **MCAR/MAR/MNAR** 중 어디에 해당하는지 판단하는 것이 중요합니다

## 2.3 기초 통계량 분석

수치형 변수의 평균, 중앙값, 분산, 사분위수 등 기초 통계량을 확인합니다.

In [ ]:
# === 수치형 변수 기초 통계량 ===
desc = df[numeric_cols].describe().T
desc['median'] = df[numeric_cols].median()
desc['skew'] = df[numeric_cols].skew().round(3)
desc['kurtosis'] = df[numeric_cols].kurtosis().round(3)
desc['IQR'] = desc['75%'] - desc['25%']
desc['CV(%)'] = (desc['std'] / desc['mean'] * 100).round(2)

# 컬럼 순서 정리
desc = desc[['count', 'mean', 'median', 'std', 'CV(%)', 'min', '25%', '50%', '75%', 'max', 'IQR', 'skew', 'kurtosis']]
desc.columns = ['N', '평균', '중앙값', '표준편차', 'CV(%)', '최솟값', 'Q1', 'Q2', 'Q3', '최댓값', 'IQR', '왜도', '첨도']
desc

### 해석

| 변수 | 주요 특징 |
|------|----------|
| **duration** | 평균 ~21개월, 오른쪽 꼬리(양의 왜도) → 단기 대출이 주류 |
| **credit_amount** | 평균 ~3,271 DM, 매우 높은 변동계수(CV) → 대출 규모 편차가 큼 |
| **age** | 평균 ~35세, 양의 왜도 → 젊은 층이 상대적으로 많음 |
| **installment_commitment** | 1~4 범위, 가처분소득 대비 할부 비율 |
| **existing_credits** | 대부분 1~2개, 최대 4개 → 다중 채무자 존재 |
| **num_dependents** | 대부분 1명, 최대 2명 → 부양가족 수가 적음 |

## 2.4 데이터 정제 (Cleaning)

중복 제거, 자료형 변환, 이상치(Outlier) 탐지 및 처리를 수행합니다.

In [ ]:
# === 중복 레코드 검사 ===
dup_count = df.duplicated().sum()
print(f'중복 레코드 수: {dup_count}')
if dup_count > 0:
    print(f'  → 중복 제거 전: {len(df)}행')
    df = df.drop_duplicates().reset_index(drop=True)
    print(f'  → 중복 제거 후: {len(df)}행')
else:
    print('✅ 중복 레코드 없음')

In [ ]:
# === 이상치 탐지 (IQR 방법) ===
print('=' * 60)
print('이상치 탐지 결과 (IQR 방법: Q1 - 1.5*IQR ~ Q3 + 1.5*IQR)')
print('=' * 60)

outlier_summary = []
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    outlier_summary.append({
        '변수': col,
        'Q1': Q1,
        'Q3': Q3,
        'IQR': IQR,
        '하한': lower,
        '상한': upper,
        '이상치 수': len(outliers),
        '이상치 비율(%)': round(len(outliers) / len(df) * 100, 2)
    })

outlier_df = pd.DataFrame(outlier_summary)
outlier_df

In [ ]:
# === 이상치 시각화: Box Plot ===
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    bp = axes[i].boxplot(df[col].dropna(), patch_artist=True, vert=True,
                         boxprops=dict(facecolor='#3498db', alpha=0.7),
                         medianprops=dict(color='red', linewidth=2))
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].set_ylabel('값')

# 빈 축 제거
for j in range(len(numeric_cols), len(axes)):
    axes[j].set_visible(False)

fig.suptitle('수치형 변수 Box Plot — 이상치 탐지', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 해석

- **credit_amount**: 가장 많은 이상치 보유 → 고액 대출 건이 존재하나, 도메인 특성상 자연스러운 현상
- **duration**: 장기 대출(60개월 이상)이 이상치로 탐지됨
- **age**: 고령 신청자가 이상치로 탐지되나, 실제 유효한 데이터
- **이상치 처리 전략**: 금융 데이터의 특성상, 이상치가 실제 의미 있는 데이터일 가능성이 높으므로 **제거하지 않고 유지**합니다. 다만, 이후 분석에서 이상치의 영향을 인지하고 해석에 반영합니다.

---

# 3. 변수별 개별 특성 분석 (Univariate Analysis)

## 3.1 수치형 변수 분석

각 수치형 변수의 분포 형태(왜도, 첨도), 히스토그램+KDE, 정규성 검정을 수행합니다.

In [ ]:
# === 왜도 & 첨도 분석 ===
skew_kurt = pd.DataFrame({
    '변수': numeric_cols,
    '왜도(Skewness)': [df[c].skew() for c in numeric_cols],
    '첨도(Kurtosis)': [df[c].kurtosis() for c in numeric_cols],
    '분포 형태': [''] * len(numeric_cols)
})

for idx, row in skew_kurt.iterrows():
    sk = row['왜도(Skewness)']
    if abs(sk) < 0.5:
        shape = '대칭에 가까움'
    elif sk > 0:
        shape = '오른쪽 꼬리 (양의 왜도)'
    else:
        shape = '왼쪽 꼬리 (음의 왜도)'
    skew_kurt.at[idx, '분포 형태'] = shape

skew_kurt = skew_kurt.round(3)
skew_kurt

In [ ]:
# === 히스토그램 + KDE ===
fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    ax = axes[i]
    
    # 히스토그램 + KDE
    sns.histplot(df[col], bins=30, kde=True, color='#3498db', alpha=0.6, ax=ax,
                 edgecolor='white', linewidth=0.5)
    
    # 평균, 중앙값 표시
    mean_val = df[col].mean()
    median_val = df[col].median()
    ax.axvline(mean_val, color='red', linestyle='--', linewidth=1.5, label=f'평균: {mean_val:.1f}')
    ax.axvline(median_val, color='green', linestyle='-.', linewidth=1.5, label=f'중앙값: {median_val:.1f}')
    
    ax.set_title(f'{col}\n(왜도={df[col].skew():.2f}, 첨도={df[col].kurtosis():.2f})',
                 fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)

for j in range(len(numeric_cols), len(axes)):
    axes[j].set_visible(False)

fig.suptitle('수치형 변수 분포 — 히스토그램 + KDE', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# === 정규성 검정 (Shapiro-Wilk) ===
print('=' * 70)
print('정규성 검정 (Shapiro-Wilk Test, H0: 정규분포를 따른다)')
print('=' * 70)

normality_results = []
for col in numeric_cols:
    # Shapiro-Wilk 검정 (n > 5000이면 샘플링)
    sample = df[col].dropna()
    if len(sample) > 5000:
        sample = sample.sample(5000, random_state=42)
    stat, p_value = shapiro(sample)
    normality_results.append({
        '변수': col,
        'W 통계량': round(stat, 4),
        'p-value': f'{p_value:.2e}',
        '정규성 (α=0.05)': '정규분포' if p_value > 0.05 else '비정규분포',
        '판정': '✅ 채택' if p_value > 0.05 else '❌ 기각'
    })

norm_df = pd.DataFrame(normality_results)
norm_df

### 해석

- **모든 수치형 변수가 정규분포를 따르지 않음** (Shapiro-Wilk p < 0.05)
- 특히 `credit_amount`, `duration`은 강한 양의 왜도 → 로그 변환 시 정규성에 가까워질 가능성
- `installment_commitment`, `existing_credits`, `num_dependents`는 이산적 특성 (정수값)
- **통계 검정 시 비모수 검정(Mann-Whitney U 등)을 사용**해야 함

## 3.2 범주형 변수 분석

각 범주형 변수의 클래스별 빈도수 및 비율, 희소 클래스(Rare labels) 존재 여부를 확인합니다.

In [ ]:
# === 타겟 변수 분포 ===
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 빈도 바차트
class_counts = df[target].value_counts()
bars = axes[0].bar(class_counts.index, class_counts.values,
                   color=[COLORS.get(x, '#95a5a6') for x in class_counts.index],
                   edgecolor='white', linewidth=2)
for bar, val in zip(bars, class_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                 f'{val}\n({val/len(df)*100:.1f}%)', ha='center', fontsize=12, fontweight='bold')
axes[0].set_title('타겟 변수 (class) 분포', fontsize=13, fontweight='bold')
axes[0].set_ylabel('빈도')

# 파이차트
axes[1].pie(class_counts.values, labels=class_counts.index,
            autopct='%1.1f%%', colors=CLASS_PALETTE,
            startangle=90, textprops={'fontsize': 13, 'fontweight': 'bold'},
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('타겟 변수 비율', fontsize=13, fontweight='bold')

plt.suptitle('타겟 변수 (class) — Good vs Bad', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'클래스 비율 — Good: {class_counts.get("good", 0)/len(df)*100:.1f}%, Bad: {class_counts.get("bad", 0)/len(df)*100:.1f}%')
print(f'불균형 비율: {class_counts.max() / class_counts.min():.2f} : 1')

In [ ]:
# === 범주형 변수 빈도 분석 (전체 서브플롯) ===
n_cats = len(categorical_cols)
n_rows = (n_cats + 2) // 3

fig, axes = plt.subplots(n_rows, 3, figsize=(20, 5 * n_rows))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    ax = axes[i]
    order = df[col].value_counts().index
    counts = df[col].value_counts()
    
    bars = ax.barh(range(len(order)), counts.values, color=sns.color_palette('Set2', len(order)),
                   edgecolor='white', linewidth=0.5)
    ax.set_yticks(range(len(order)))
    ax.set_yticklabels(order, fontsize=9)
    ax.set_title(f'{col} (고유값: {len(order)}개)', fontsize=11, fontweight='bold')
    ax.set_xlabel('빈도')
    
    # 빈도 레이블
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
                f'{val} ({val/len(df)*100:.1f}%)', va='center', fontsize=8)

for j in range(n_cats, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('범주형 변수 빈도 분포', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# === 희소 클래스(Rare Labels) 탐지 ===
print('=' * 60)
print('희소 클래스 탐지 (빈도 < 5%인 범주)')
print('=' * 60)

rare_labels = []
for col in categorical_cols:
    freq = df[col].value_counts(normalize=True) * 100
    rare = freq[freq < 5]
    if len(rare) > 0:
        for label, pct in rare.items():
            rare_labels.append({
                '변수': col,
                '범주': label,
                '빈도': df[col].value_counts()[label],
                '비율(%)': round(pct, 2)
            })

if rare_labels:
    rare_df = pd.DataFrame(rare_labels)
    print(f'\n총 {len(rare_df)}개의 희소 범주 발견:')
    display(rare_df)
else:
    print('\n희소 클래스 없음')

### 해석

- **타겟 불균형**: Good 70% vs Bad 30% → 약 2.3:1 비율의 불균형 존재
- **주요 범주 패턴**:
  - `checking_status`: 'no checking'이 가장 많음 → 당좌예금 미보유자가 다수
  - `purpose`: radio/tv, new car가 상위 → 소비재 목적 대출이 다수
  - `housing`: own(자가)이 과반 이상
  - `foreign_worker`: yes가 압도적 다수 (96%+)
- **희소 클래스**: `purpose`의 일부 범주(vacation, domestic appliance 등), `personal_status`의 일부가 5% 미만

## 3.3 파생 변수 생성 (Feature Engineering)

연속형 변수를 구간화(Binning)하여 새로운 범주형 변수를 생성합니다. 이를 통해 패턴을 더 직관적으로 파악할 수 있습니다.

In [ ]:
# === 파생 변수 생성: 구간화(Binning) ===

# 1) age 구간화
df['age_group'] = pd.cut(df['age'], bins=[0, 25, 35, 45, 55, 100],
                         labels=['청년(~25)', '초기성인(26~35)', '중년(36~45)', '장년(46~55)', '고령(56~)'])

# 2) credit_amount 구간화
df['credit_group'] = pd.cut(df['credit_amount'],
                            bins=[0, 1000, 2500, 5000, 10000, 20000],
                            labels=['소액(~1K)', '중소(1K~2.5K)', '중간(2.5K~5K)', '고액(5K~10K)', '초고액(10K~)'])

# 3) duration 구간화
df['duration_group'] = pd.cut(df['duration'],
                              bins=[0, 12, 24, 36, 72],
                              labels=['단기(~12개월)', '중기(13~24개월)', '장기(25~36개월)', '초장기(37개월~)'])

print('파생 변수 생성 완료:')
for col in ['age_group', 'credit_group', 'duration_group']:
    print(f'\n[{col}]')
    print(df[col].value_counts().sort_index())

In [ ]:
# === 파생 변수 시각화 ===
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, col in enumerate(['age_group', 'credit_group', 'duration_group']):
    ct = pd.crosstab(df[col], df[target], normalize='index') * 100
    ct.plot(kind='bar', stacked=True, color=CLASS_PALETTE, ax=axes[i],
            edgecolor='white', linewidth=0.5)
    axes[i].set_title(f'{col}별 Good/Bad 비율', fontsize=12, fontweight='bold')
    axes[i].set_ylabel('비율 (%)')
    axes[i].set_xlabel('')
    axes[i].legend(title='class', loc='upper right')
    axes[i].tick_params(axis='x', rotation=45)
    # 비율 레이블 추가
    for container in axes[i].containers:
        axes[i].bar_label(container, fmt='%.1f%%', label_type='center', fontsize=8)

plt.suptitle('파생 변수별 신용등급 분포', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 해석

- **연령 구간**: 청년층(~25세)에서 Bad 비율이 상대적으로 높고, 중년/장년층은 Good 비율이 높은 경향
- **대출 금액 구간**: 초고액 대출(10K+ DM)에서 Bad 비율이 뚜렷이 증가 → 가설 H2를 지지하는 예비 증거
- **대출 기간 구간**: 장기/초장기 대출일수록 Bad 비율 증가 → 대출 기간이 리스크 지표

---

# 4. 상관관계 및 관계 분석 (Multivariate Analysis)

## 4.1 수치형 변수 간 상관관계

상관 계수 히트맵을 통해 다중공선성(Multicollinearity)을 확인하고,  
주요 변수 간 산점도와 추세선으로 관계 패턴을 탐색합니다.

In [ ]:
# === 상관 계수 히트맵 ===
corr_matrix = df[numeric_cols].corr()

# 마스크 (상삼각 행렬만 표시)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.3f',
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=1, linecolor='white',
            cbar_kws={'label': '상관계수', 'shrink': 0.8}, ax=ax)
ax.set_title('수치형 변수 상관계수 히트맵 (Pearson)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# 강한 상관관계 추출
print('\n강한 상관관계 (|r| > 0.3):')
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.3:
            print(f'  {corr_matrix.columns[i]} ↔ {corr_matrix.columns[j]}: r = {r:.3f}')

In [ ]:
# === 주요 변수 산점도 + 추세선 ===
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1) duration vs credit_amount
for cls, color in COLORS.items():
    subset = df[df[target] == cls]
    axes[0].scatter(subset['duration'], subset['credit_amount'], 
                    c=color, alpha=0.5, s=30, label=cls, edgecolor='white', linewidth=0.3)
# 전체 추세선
z = np.polyfit(df['duration'], df['credit_amount'], 1)
p = np.poly1d(z)
x_line = np.linspace(df['duration'].min(), df['duration'].max(), 100)
axes[0].plot(x_line, p(x_line), 'k--', linewidth=2, alpha=0.7, label=f'추세선 (기울기={z[0]:.1f})')
axes[0].set_xlabel('대출 기간 (duration)')
axes[0].set_ylabel('대출 금액 (credit_amount)')
axes[0].set_title('Duration vs Credit Amount', fontsize=12, fontweight='bold')
axes[0].legend()

# 2) age vs credit_amount
for cls, color in COLORS.items():
    subset = df[df[target] == cls]
    axes[1].scatter(subset['age'], subset['credit_amount'],
                    c=color, alpha=0.5, s=30, label=cls, edgecolor='white', linewidth=0.3)
axes[1].set_xlabel('나이 (age)')
axes[1].set_ylabel('대출 금액 (credit_amount)')
axes[1].set_title('Age vs Credit Amount', fontsize=12, fontweight='bold')
axes[1].legend()

# 3) duration vs age
for cls, color in COLORS.items():
    subset = df[df[target] == cls]
    axes[2].scatter(subset['duration'], subset['age'],
                    c=color, alpha=0.5, s=30, label=cls, edgecolor='white', linewidth=0.3)
axes[2].set_xlabel('대출 기간 (duration)')
axes[2].set_ylabel('나이 (age)')
axes[2].set_title('Duration vs Age', fontsize=12, fontweight='bold')
axes[2].legend()

plt.suptitle('주요 수치형 변수 산점도 (타겟별 색상)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 해석

- **duration ↔ credit_amount**: 양의 상관관계 → 대출 기간이 길수록 대출 금액도 큼 (가설 H4 지지)
- **다중공선성**: 대부분의 변수 간 상관관계가 약함 (|r| < 0.3) → 심각한 다중공선성 문제 없음
- **타겟 기반 패턴**: 산점도에서 Bad(빨강)가 고액·장기 대출 영역에 집중되는 경향이 보임

## 4.2 타겟 변수 기반 심층 분석

독립 변수들이 종속 변수(class)에 미치는 영향력을 시각화합니다.

In [ ]:
# === 수치형 변수 × 타겟: Box Plot + Violin Plot ===
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for i, col in enumerate(numeric_cols):
    row, col_idx = divmod(i, 4)
    ax = axes[row][col_idx]
    
    # Violin + Box 조합
    parts = ax.violinplot([df[df[target] == 'good'][col].dropna(),
                           df[df[target] == 'bad'][col].dropna()],
                          positions=[1, 2], showmedians=True, showextrema=False)
    
    for idx, pc in enumerate(parts['bodies']):
        pc.set_facecolor(CLASS_PALETTE[idx])
        pc.set_alpha(0.4)
    parts['cmedians'].set_color('black')
    
    # Box plot 오버레이
    bp = ax.boxplot([df[df[target] == 'good'][col].dropna(),
                     df[df[target] == 'bad'][col].dropna()],
                    positions=[1, 2], widths=0.2, patch_artist=True,
                    showfliers=False)
    for patch, color in zip(bp['boxes'], CLASS_PALETTE):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    
    ax.set_xticks([1, 2])
    ax.set_xticklabels(['Good', 'Bad'])
    ax.set_title(col, fontsize=11, fontweight='bold')
    ax.set_ylabel('값')

# 빈 축 제거
for j in range(len(numeric_cols), 8):
    row, col_idx = divmod(j, 4)
    axes[row][col_idx].set_visible(False)

fig.suptitle('수치형 변수 × 신용등급: Violin + Box Plot', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# === 범주형 변수 × 타겟: Stacked Bar Chart (주요 6개 변수) ===
key_cats = ['checking_status', 'credit_history', 'purpose', 'savings_status', 'employment', 'housing']

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

for i, col in enumerate(key_cats):
    ax = axes[i]
    ct = pd.crosstab(df[col], df[target], normalize='index') * 100
    ct = ct.reindex(columns=['good', 'bad'])
    
    ct.plot(kind='barh', stacked=True, color=CLASS_PALETTE, ax=ax,
            edgecolor='white', linewidth=0.5)
    
    ax.set_title(f'{col}별 Good/Bad 비율', fontsize=12, fontweight='bold')
    ax.set_xlabel('비율 (%)')
    ax.set_ylabel('')
    ax.legend(title='class', loc='lower right')
    ax.set_xlim(0, 100)
    
    # Bad 비율 레이블
    for idx, (label, row) in enumerate(ct.iterrows()):
        bad_pct = row.get('bad', 0)
        ax.text(98, idx, f'{bad_pct:.1f}%', va='center', ha='right',
                fontsize=9, fontweight='bold', color='white')

plt.suptitle('주요 범주형 변수 × 신용등급 (Stacked Bar)', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 해석

**수치형 변수 (Violin + Box Plot):**
- `duration`: Bad 그룹의 중앙값이 Good보다 높음 → 장기 대출이 부실 위험 증가
- `credit_amount`: Bad 그룹이 더 넓은 분포와 높은 중앙값 → 고액 대출이 리스크 요인
- `age`: Good 그룹이 약간 더 높은 나이 분포 → 경험/안정성이 신용에 기여

**범주형 변수 (Stacked Bar):**
- `checking_status`: '<0' (마이너스 잔고)에서 Bad 비율이 가장 높음 → 강력한 예측 변수
- `credit_history`: 'critical/other existing credit'에서 오히려 Bad 비율이 낮음 → 역설적 패턴
- `savings_status`: '<100' (소액 저축)에서 Bad 비율 높음 → 저축이 안전 완충재

## 4.3 세그먼트별 비교 분석

특정 조건(예: VIP 고객 vs 일반 고객)에 따른 행동 패턴 차이를 분석합니다.

In [ ]:
# === 세그먼트 정의 ===
# VIP 세그먼트: 당좌예금 200+ AND 저축 500+ AND 자가 보유
df['segment'] = 'Regular'
vip_mask = (
    (df['checking_status'].isin(['>=200'])) & 
    (df['savings_status'].isin(['500<=X<1000', '>=1000'])) &
    (df['housing'] == 'own')
)
df.loc[vip_mask, 'segment'] = 'VIP'

# 고위험 세그먼트: 당좌예금 마이너스 AND 저축 100 미만
high_risk_mask = (
    (df['checking_status'] == '<0') &
    (df['savings_status'] == '<100')
)
df.loc[high_risk_mask, 'segment'] = 'High-Risk'

print('세그먼트별 분포:')
print(df['segment'].value_counts())
print()
print('세그먼트별 Bad 비율:')
print(df.groupby('segment')[target].apply(lambda x: (x == 'bad').mean() * 100).round(2).to_string())

In [ ]:
# === 세그먼트별 비교 시각화 ===
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1) 세그먼트별 Bad 비율
seg_bad = df.groupby('segment')[target].apply(lambda x: (x == 'bad').mean() * 100)
colors_seg = {'VIP': '#2ecc71', 'Regular': '#3498db', 'High-Risk': '#e74c3c'}
bars = axes[0].bar(seg_bad.index, seg_bad.values,
                   color=[colors_seg.get(s, '#95a5a6') for s in seg_bad.index],
                   edgecolor='white', linewidth=2)
for bar, val in zip(bars, seg_bad.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val:.1f}%', ha='center', fontsize=12, fontweight='bold')
axes[0].set_title('세그먼트별 Bad 비율', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Bad 비율 (%)')

# 2) 세그먼트별 대출 금액 분포
for seg in ['VIP', 'Regular', 'High-Risk']:
    subset = df[df['segment'] == seg]['credit_amount']
    if len(subset) > 0:
        axes[1].hist(subset, bins=20, alpha=0.5, label=f'{seg} (n={len(subset)})',
                     color=colors_seg.get(seg, '#95a5a6'), edgecolor='white')
axes[1].set_title('세그먼트별 대출 금액 분포', fontsize=12, fontweight='bold')
axes[1].set_xlabel('대출 금액 (credit_amount)')
axes[1].set_ylabel('빈도')
axes[1].legend()

# 3) 세그먼트별 평균 프로필
seg_profile = df.groupby('segment')[['duration', 'credit_amount', 'age']].mean()
seg_profile.plot(kind='bar', ax=axes[2], color=['#3498db', '#e74c3c', '#2ecc71'],
                 edgecolor='white', linewidth=0.5)
axes[2].set_title('세그먼트별 평균 프로필', fontsize=12, fontweight='bold')
axes[2].set_ylabel('평균값')
axes[2].tick_params(axis='x', rotation=0)
axes[2].legend(loc='upper right')

plt.suptitle('세그먼트 비교 분석 (VIP vs Regular vs High-Risk)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# === 고액 vs 저액 대출 비교 ===
median_amount = df['credit_amount'].median()
df['loan_size'] = df['credit_amount'].apply(lambda x: '고액 대출' if x > median_amount else '저액 대출')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bad 비율 비교
loan_bad = df.groupby('loan_size')[target].apply(lambda x: (x == 'bad').mean() * 100)
bars = axes[0].bar(loan_bad.index, loan_bad.values,
                   color=['#e74c3c', '#2ecc71'], edgecolor='white', linewidth=2)
for bar, val in zip(bars, loan_bad.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val:.1f}%', ha='center', fontsize=13, fontweight='bold')
axes[0].set_title(f'대출 규모별 Bad 비율 (기준: 중앙값 {median_amount:.0f} DM)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Bad 비율 (%)')

# 대출 목적 비교
ct_purpose = pd.crosstab(df['purpose'], df['loan_size'], normalize='columns') * 100
ct_purpose.plot(kind='barh', ax=axes[1], color=['#e74c3c', '#2ecc71'],
                edgecolor='white', linewidth=0.5)
axes[1].set_title('대출 규모별 목적 분포', fontsize=12, fontweight='bold')
axes[1].set_xlabel('비율 (%)')
axes[1].set_ylabel('')

plt.suptitle('고액 vs 저액 대출 비교', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 해석

**세그먼트 비교:**
- **VIP 세그먼트**: 예상대로 Bad 비율이 가장 낮음 → 금융 안정성이 신용의 강력한 예측 인자
- **High-Risk 세그먼트**: Bad 비율이 VIP 대비 크게 높음 → 당좌예금 마이너스 + 저축 부족이 복합 리스크

**대출 규모 비교:**
- 고액 대출에서 Bad 비율이 더 높음 → 가설 H2를 지지
- 고액 대출의 주요 목적: new car, business 등 → 목적별 리스크 차이 존재

---

# 5. 핵심 인사이트 및 가설 검정 (Key Insights)

## 5.1 주요 패턴 발견

앞선 시각화 결과에서 도출된 결정적 특징을 요약합니다.

### 핵심 패턴 요약

| 순위 | 패턴 | 근거 |
|------|------|------|
| 1 | **당좌예금 상태가 가장 강력한 리스크 지표** | checking_status '<0'에서 Bad 비율 최고 |
| 2 | **대출 기간이 길수록 부실 위험 증가** | duration과 Bad 비율 양의 관계 |
| 3 | **대출 금액이 클수록 부실 위험 증가** | credit_amount와 Bad 비율 양의 관계 |
| 4 | **저축/자산이 리스크 완충재** | savings_status가 높을수록 Good 비율 증가 |
| 5 | **젊은 층의 상대적 고위험** | age_group 청년에서 Bad 비율 높음 |

## 5.2 가설 검증 결과

서론에서 설정한 가설을 통계적으로 검증합니다.

In [ ]:
# ====================================================================
# 가설 1: checking_status가 신용등급에 유의미한 영향을 미친다
# 검정 방법: 카이제곱 독립성 검정 (범주형 × 범주형)
# ====================================================================
print('=' * 70)
print('가설 H1: checking_status가 신용등급(class)에 유의미한 영향을 미친다')
print('검정 방법: 카이제곱 독립성 검정 (Chi-Square Test of Independence)')
print('=' * 70)

ct_h1 = pd.crosstab(df['checking_status'], df[target])
chi2, p_val, dof, expected = chi2_contingency(ct_h1)

# Cramér's V (효과 크기)
n = ct_h1.sum().sum()
k = min(ct_h1.shape) - 1
cramers_v = np.sqrt(chi2 / (n * k))

print(f'\n교차표:')
display(ct_h1)
print(f'\n카이제곱 통계량: χ² = {chi2:.4f}')
print(f'자유도: {dof}')
print(f'p-value: {p_val:.2e}')
print(f"Cramér's V (효과 크기): {cramers_v:.4f}")
print(f'\n결론: p-value = {p_val:.2e} < 0.05 → ' + 
      ('❌ H0 기각 → checking_status는 신용등급에 유의미한 영향을 미침 ✅' if p_val < 0.05 
       else '✅ H0 채택 → 유의미한 영향 없음'))

In [ ]:
# ====================================================================
# 가설 2: credit_amount가 클수록 bad 비율이 증가한다
# 검정 방법: Mann-Whitney U 검정 (비모수, 두 그룹 비교)
# ====================================================================
print('=' * 70)
print('가설 H2: credit_amount가 클수록 Bad 비율이 증가한다')
print('검정 방법: Mann-Whitney U 검정 (비모수)')
print('=' * 70)

good_amount = df[df[target] == 'good']['credit_amount']
bad_amount = df[df[target] == 'bad']['credit_amount']

stat_mw, p_val_mw = mannwhitneyu(good_amount, bad_amount, alternative='less')

# 효과 크기 (rank-biserial correlation)
n1, n2 = len(good_amount), len(bad_amount)
r_rb = 1 - (2 * stat_mw) / (n1 * n2)

print(f'\nGood 그룹 — 평균: {good_amount.mean():.1f}, 중앙값: {good_amount.median():.1f}')
print(f'Bad 그룹  — 평균: {bad_amount.mean():.1f}, 중앙값: {bad_amount.median():.1f}')
print(f'\nMann-Whitney U 통계량: U = {stat_mw:.1f}')
print(f'p-value (단측): {p_val_mw:.2e}')
print(f'효과 크기 (rank-biserial r): {r_rb:.4f}')
print(f'\n결론: p-value = {p_val_mw:.2e} < 0.05 → ' +
      ('❌ H0 기각 → Bad 그룹의 대출 금액이 유의미하게 더 큼 ✅' if p_val_mw < 0.05
       else '✅ H0 채택 → 두 그룹 간 유의미한 차이 없음'))

In [ ]:
# ====================================================================
# 가설 3: employment 기간이 길수록 good 비율이 높다
# 검정 방법: 카이제곱 독립성 검정
# ====================================================================
print('=' * 70)
print('가설 H3: employment 기간이 길수록 Good 비율이 높다')
print('검정 방법: 카이제곱 독립성 검정')
print('=' * 70)

ct_h3 = pd.crosstab(df['employment'], df[target])
chi2_h3, p_val_h3, dof_h3, expected_h3 = chi2_contingency(ct_h3)

k_h3 = min(ct_h3.shape) - 1
cramers_v_h3 = np.sqrt(chi2_h3 / (n * k_h3))

# 고용 기간별 Good 비율
emp_good_rate = pd.crosstab(df['employment'], df[target], normalize='index')['good'] * 100

print(f'\n고용 기간별 Good 비율:')
for emp, rate in emp_good_rate.items():
    print(f'  {emp}: {rate:.1f}%')

print(f'\n카이제곱 통계량: χ² = {chi2_h3:.4f}')
print(f'p-value: {p_val_h3:.2e}')
print(f"Cramér's V: {cramers_v_h3:.4f}")
print(f'\n결론: p-value = {p_val_h3:.2e} ' +
      (f'< 0.05 → ❌ H0 기각 → employment는 신용등급에 유의미한 영향 ✅' if p_val_h3 < 0.05
       else f'>= 0.05 → ✅ H0 채택 → 유의미한 영향 없음'))

In [ ]:
# ====================================================================
# 가설 4: duration과 credit_amount는 양의 상관관계
# 검정 방법: Spearman 상관계수 (비모수)
# ====================================================================
print('=' * 70)
print('가설 H4: duration과 credit_amount는 양의 상관관계를 보인다')
print('검정 방법: Spearman 순위 상관분석')
print('=' * 70)

rho, p_val_h4 = stats.spearmanr(df['duration'], df['credit_amount'])

print(f'\nSpearman ρ = {rho:.4f}')
print(f'p-value: {p_val_h4:.2e}')
print(f'\n결론: ρ = {rho:.4f}, p-value = {p_val_h4:.2e} → ' +
      ('유의미한 양의 상관관계 존재 ✅' if (p_val_h4 < 0.05 and rho > 0)
       else '유의미한 양의 상관관계 없음'))

In [ ]:
# === 가설 검증 종합 요약 ===
print('\n' + '=' * 70)
print('가설 검증 종합 요약')
print('=' * 70)

hypothesis_summary = pd.DataFrame([
    {'가설': 'H1', '내용': 'checking_status → class 영향',
     '검정 방법': '카이제곱', 'p-value': f'{p_val:.2e}', '결과': '채택' if p_val < 0.05 else '기각'},
    {'가설': 'H2', '내용': 'credit_amount ↑ → Bad ↑',
     '검정 방법': 'Mann-Whitney U', 'p-value': f'{p_val_mw:.2e}', '결과': '채택' if p_val_mw < 0.05 else '기각'},
    {'가설': 'H3', '내용': 'employment ↑ → Good ↑',
     '검정 방법': '카이제곱', 'p-value': f'{p_val_h3:.2e}', '결과': '채택' if p_val_h3 < 0.05 else '기각'},
    {'가설': 'H4', '내용': 'duration ↔ credit_amount 양의 상관',
     '검정 방법': 'Spearman', 'p-value': f'{p_val_h4:.2e}', '결과': '채택' if (p_val_h4 < 0.05 and rho > 0) else '기각'},
])
hypothesis_summary

## 5.3 예상치 못한 발견 (Unexpected Findings)

초기 가설 외에 EDA 과정에서 새롭게 발견된 특이 사항들입니다.

In [ ]:
# === Unexpected Finding 1: credit_history의 역설적 패턴 ===
print('=' * 70)
print('Unexpected Finding 1: credit_history의 역설적 패턴')
print('=' * 70)

ch_ct = pd.crosstab(df['credit_history'], df[target], normalize='index') * 100
ch_ct = ch_ct.reindex(columns=['good', 'bad'])
print('\ncredit_history별 Bad 비율:')
print(ch_ct['bad'].sort_values(ascending=False).round(2).to_string())
print('\n→ "critical/other existing credit"에서 Bad 비율이 가장 낮음!')
print('  이는 기존에 다수의 신용 이력이 있는 고객이 오히려 상환 능력이 검증되었기 때문으로 해석됨')

In [ ]:
# === Unexpected Finding 2: foreign_worker의 편향 ===
print('=' * 70)
print('Unexpected Finding 2: foreign_worker의 극단적 편향')
print('=' * 70)

fw_dist = df['foreign_worker'].value_counts()
fw_bad = pd.crosstab(df['foreign_worker'], df[target], normalize='index') * 100

print(f'\nforeign_worker 분포:')
for val, cnt in fw_dist.items():
    print(f'  {val}: {cnt}건 ({cnt/len(df)*100:.1f}%)')
print(f'\n→ yes가 {fw_dist.get("yes", 0)/len(df)*100:.1f}%로 압도적 다수')
print('  이 변수는 분류 모델에서 유의미한 예측력을 가지기 어려울 수 있음 (편향 과다)')

In [ ]:
# === Unexpected Finding 3: 나이와 대출 목적의 교차 패턴 ===
print('=' * 70)
print('Unexpected Finding 3: 연령대별 대출 목적 차이')
print('=' * 70)

age_purpose = pd.crosstab(df['age_group'], df['purpose'], normalize='index') * 100
top_purposes = df['purpose'].value_counts().head(5).index

fig, ax = plt.subplots(figsize=(12, 6))
age_purpose[top_purposes].plot(kind='bar', ax=ax, edgecolor='white', linewidth=0.5)
ax.set_title('연령대별 상위 5개 대출 목적 분포', fontsize=13, fontweight='bold')
ax.set_ylabel('비율 (%)')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=45)
ax.legend(title='대출 목적', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

print('\n→ 청년층: radio/tv, new car 등 소비재 중심')
print('  중장년층: business, furniture/equipment 등 생산적 목적 증가')
print('  이는 연령에 따른 재정적 필요의 변화를 반영')

### 예상치 못한 발견 요약

| 번호 | 발견 | 의미 |
|------|------|------|
| 1 | credit_history가 'critical'인 경우 오히려 Bad 비율 낮음 | **상환 이력 자체가 신뢰 지표** — 단순히 '과거 문제 = 현재 위험'이 아님 |
| 2 | foreign_worker 변수의 극단적 편향 (96%+ yes) | **모델링 시 해당 변수의 유용성이 제한적** — 제거 또는 재설계 고려 |
| 3 | 연령대별 대출 목적 패턴이 뚜렷이 다름 | **나이 × 목적 교호작용(interaction)** 이 리스크 예측에 유용할 가능성 |

---

# 6. 결론 및 향후 방향 (Conclusion & Recommendation)

## 6.1 분석 요약

### 전체 EDA 프로세스 핵심 요약

| 단계 | 핵심 결과 |
|------|----------|
| **데이터 프로파일링** | 1,000건 × 21변수, 결측치 없음, 이상치는 도메인 특성으로 유지 |
| **단변량 분석** | 모든 수치형 비정규분포, 타겟 불균형(70:30), 희소 범주 존재 |
| **다변량 분석** | duration↔credit_amount 양의 상관, checking_status가 가장 강력한 예측 변수 |
| **가설 검정** | 4개 가설 모두 통계적으로 유의미한 결과 도출 |
| **예상치 못한 발견** | credit_history 역설, foreign_worker 편향, 나이×목적 교호작용 |

### 신용등급 예측에 중요한 변수 (Top 5)

1. **checking_status** — 당좌예금 상태 (가장 강력한 예측 변수)
2. **duration** — 대출 기간
3. **credit_amount** — 대출 금액
4. **savings_status** — 저축 상태
5. **age** — 나이

## 6.2 비즈니스 제언 (Action Plan)

EDA 결과를 바탕으로 한 실행 가능한 전략을 제안합니다.

### 대출 심사 기준 제안

| 구분 | 제안 | 근거 |
|------|------|------|
| **1차 스크리닝** | 당좌예금 상태 기반 사전 분류 | checking_status가 가장 강력한 예측 변수 |
| **리스크 가중** | 대출 기간 36개월+ 건에 추가 심사 | 장기 대출의 Bad 비율 유의미하게 높음 |
| **금액 한도** | 저축 대비 대출 비율(LTV) 기준 설정 | savings_status와 credit_amount의 교차 효과 |
| **연령 보정** | 청년층(~25세)에 추가 보증/담보 요구 | 연령 구간별 Bad 비율 차이 유의미 |
| **세그먼트 전략** | VIP(금융 안정)/High-Risk(마이너스 잔고+저축 부족) 차별 관리 | 세그먼트별 Bad 비율 격차 뚜렷 |

### 비대칭 비용을 고려한 의사결정

- FN 비용이 5배이므로, **"의심스러우면 거절"** 전략이 총 비용을 최소화
- 특히 checking_status '<0' + 고액 대출 + 장기 → 자동 거절 또는 2차 심사 대상
- 반면, VIP 세그먼트는 빠른 승인 프로세스 적용 가능 (기회비용 최소화)

## 6.3 한계점 및 추후 과제

### 데이터 한계

| 한계 | 설명 | 영향 |
|------|------|------|
| **시대적 한계** | 1994년 독일 데이터 → 현재 금융 환경과 다름 | 변수 중요도가 현대에 달라질 가능성 |
| **샘플 크기** | 1,000건은 ML 모델 학습에 소규모 | 과적합(overfitting) 위험, 교차검증 필수 |
| **샘플링 편향** | 수집 방법 불명확, 대표성 검증 불가 | 일반화 가능성에 한계 |
| **변수 제한** | 소득, 직업 상세, 신용점수 등 핵심 정보 부재 | 예측력에 한계 |
| **foreign_worker 편향** | 96%+ yes로 변별력 부족 | 모델에서 해당 변수의 기여도 제한적 |

### 향후 머신러닝 모델 도입 방향

| 단계 | 과제 | 세부 내용 |
|------|------|----------|
| **전처리** | 범주형 인코딩 | One-hot / Target Encoding (고카디널리티 변수) |
| **전처리** | 불균형 처리 | SMOTE, Cost-sensitive learning (비용 행렬 반영) |
| **모델링** | 기준 모델(Baseline) | Logistic Regression → 해석 가능성 확보 |
| **모델링** | 앙상블 모델 | Random Forest, XGBoost → 예측력 극대화 |
| **평가** | 비용 기반 평가 | Accuracy보다 **Expected Cost** 최소화 기준으로 평가 |
| **해석** | 모델 해석 | SHAP, Feature Importance → 비즈니스 인사이트 연결 |

### 최종 결론

본 EDA를 통해 German Credit 데이터셋의 구조와 특성을 심층적으로 파악하였습니다.  
**당좌예금 상태, 대출 기간, 대출 금액**이 신용 리스크의 핵심 예측 변수임을 통계적으로 확인하였으며,  
비대칭 비용 구조를 고려한 의사결정 전략을 제안하였습니다.  
향후 이 EDA 결과를 바탕으로 Cost-sensitive ML 모델을 개발하면 더욱 정교한 신용 평가 시스템을 구축할 수 있을 것입니다.

In [ ]:
# === 분석에 사용된 파생 변수 정리 (원본 보존) ===
derived_cols = ['age_group', 'credit_group', 'duration_group', 'segment', 'loan_size']
print(f'생성된 파생 변수: {derived_cols}')
print(f'원본 데이터 크기: {df.drop(columns=derived_cols).shape}')
print(f'파생 변수 포함 크기: {df.shape}')
print('\n✅ 분석 완료!')